# Wallet (Actor) Classification on Elliptic++

End-to-end notebook for the actor-dataset wallet classification task. Everything below is the wallet task; the transactions sub-dataset is used only as a source of `tx` node features for the heterogeneous (wallet, tx) graph.

## Models
1. **Random Forest** baseline on per-`(wallet, time-step)` tabular features.
2. **Logistic Regression** baseline on the same.
3. **Temporal Transformer with GNN embedding** — swappable GCN / GraphSAGE encoder, configurable depth.
4. **Graph Attention Hawkes** (improved) — full-population CT-LSTM + dual attention, focal loss, threshold calibration.

## Data and split
- `actors_data/wallets_features.txt`: 1,268,260 per-`(address, Time step)` rows, 55 features each.
- `actors_data/wallets_classes.txt`: 822,942 per-address labels.
- `actors_data/{AddrAddr,AddrTx,TxAddr}_edgelist.txt`: heterogeneous edges (the only edges that touch the actor graph).
- `transactions_data/txs_features.csv`: tx node features (no tx labels used here).

**70/30 stratified PER-WALLET split** (`random_state=42`). Every wallet is fully on one side; every per-`(wallet, time-step)` row inherits its wallet's side. This eliminates the trivial leakage that a per-row random split would cause given that each wallet's 55-feature vector is **byte-for-byte identical across all of its time-steps** (verified directly on `wallets_features.txt`).

### What the original paper actually does
The Elmougy & Liu (KDD '23) reference notebook does `drop(columns=['Time step']).drop_duplicates()` *before* the 70/30 split (`shuffle=False`, `random_state=15`). With constant-per-wallet features the de-dupe collapses 1.27M rows to ~265K unique-wallet rows, so the paper's reported ~0.83 illicit F1 is a **per-wallet** metric, not a per-row metric. Our per-wallet stratified split is the same protocol with stratification added.

### What this means for the temporal models
Because the wallet feature file has zero per-step variation, the Temporal Transformer and Graph-Hawkes models receive an *identical* per-step token at every position — there is no temporal signal in the wallet features themselves. Time-varying signal is only available via the activation pattern of the heterogeneous edges (which `tx`/`wallet` participates at each `t`).

## Configurable hyperparameters
All hyperparameters live in the **config block** at the top of the imports cell. To swap GNN type for the Temporal Transformer's embedding, edit `EMBEDDER_CONV_TYPE`. To change embedder depth, edit `EMBEDDER_NUM_LAYERS`. To resize the Hawkes model, edit the `HAWKES_*` block.

In [3]:
!pip install -q \
  "numpy<2" \
  "torch-geometric>=2.4.0" \
  "scikit-learn>=1.3.0" \
  "pandas>=2.0.0" \
  "matplotlib>=3.7.0" \
  "seaborn>=0.12.0" \
  "pyyaml>=6.0" \
  "tqdm>=4.65.0"

In [ ]:
!pip install -q --force-reinstall \
  "numpy<2" \
  "pandas>=2.0,<2.2.3" \
  "scikit-learn>=1.3.0" \
  "torch-geometric>=2.4.0"

import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 207.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 177.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [31]:
from pathlib import Path
import warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, roc_auc_score, precision_recall_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, GCNConv, SAGEConv, GATConv
from torch_geometric.transforms import ToUndirected

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

# Mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ROOT = Path("/content/drive/MyDrive/stat175-final-project")

TRANSACTIONS_DIR = ROOT / "data"
WALLETS_DIR = ROOT / "actor_data"

# ============================================================
#  CONFIG — change here, then re-run
# ============================================================
TEST_SPLIT = 0.30
RANDOM_SEED = 42
N_TIME_STEPS = 49

# GNN embedder (used by the Temporal Transformer)
EMBEDDER_CONV_TYPE   = "sage"     # "gcn" or "sage"
EMBEDDER_NUM_LAYERS  = 3
EMBEDDER_HIDDEN_DIM  = 128
EMBEDDER_DROPOUT     = 0.4
EMBEDDER_LR          = 1e-3
EMBEDDER_EPOCHS      = 50
RUN_BOTH_EMBEDDERS   = True       # train both gcn AND sage and compare

# Temporal Transformer
TEMPORAL_D_MODEL     = 128
TEMPORAL_N_HEADS     = 4
TEMPORAL_N_LAYERS    = 3
TEMPORAL_DROPOUT     = 0.2
TEMPORAL_LR          = 1e-3
TEMPORAL_EPOCHS      = 50
TEMPORAL_BATCH_SIZE  = 1024

# Graph Attention Hawkes
HAWKES_HIDDEN_DIM    = 256
HAWKES_WINDOW        = 8
HAWKES_N_ATTN_HEADS  = 4
HAWKES_GAT_HEADS     = 2
HAWKES_BETA_DECAY_INIT = 0.2
HAWKES_FUSE_MODE     = "gated"    # "concat", "gated", "sum"
HAWKES_DROPOUT       = 0.2
HAWKES_FOCAL_GAMMA   = 2.0
HAWKES_LR            = 5e-4
HAWKES_WEIGHT_DECAY  = 5e-4
HAWKES_EPOCHS        = 50
HAWKES_TBPTT_K       = 5
USE_AMP              = True       # mixed precision (bf16) on A100
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"root={ROOT}  device={DEVICE}")

Mounted at /content/drive
root=/content/drive/MyDrive/stat175-final-project  device=cuda


## 1. Load data and build the heterogeneous graph (single cell)

We build:
- `labeled_rows`: per-`(address, Time step)` table, only rows whose wallet has a known label.
- `train_row_idx` / `test_row_idx`: row indices induced by a stratified 70/30 split at the **wallet** level (every wallet is fully train OR fully test). Every model below evaluates on **the same `test_row_idx`**.
- `X_per_row_std`: feature matrix, standardised on training rows only (no leakage).
- `hetero_graph`: static `HeteroData` with `wallet` (822K) and `tx` (204K) node types and three actor relations (`AddrAddr`, `AddrTx`, `TxAddr`); `ToUndirected()` adds reverse edges.
- Per-row index → `(wallet_idx, time_step)` and `event_train` / `event_test` boolean tensors keyed by `(wallet_idx, time_step)` — used by the Hawkes model to apply the row split at the event level.
- Wallet **node** features for the static graph are aggregated as the mean of each wallet's standardised per-step rows (gives the GNN a stable per-wallet input). Tx **node** features are standardised per-tx.

In [4]:
def map_edges(edge_df, src_col, dst_col, src_map, dst_map):
    src = edge_df[src_col].map(src_map)
    dst = edge_df[dst_col].map(dst_map)
    valid = src.notna() & dst.notna()
    return torch.tensor(
        np.stack([src[valid].astype(np.int64).values, dst[valid].astype(np.int64).values]),
        dtype=torch.long,
    )

# === Wallet labels (per address) ===
wallet_classes = pd.read_csv(WALLETS_DIR / "wallets_classes.txt")
wallet_classes["label"] = wallet_classes["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
wallet_addr_to_idx = {addr: i for i, addr in enumerate(wallet_classes["address"].values)}
N_w = len(wallet_classes)
wallet_label_per_addr = wallet_classes["label"].values

# === Wallet per-step features (all rows, labelled and unlabelled) ===
with open(WALLETS_DIR / "wallets_features.txt") as f:
    wallet_columns = f.readline().rstrip("\n").split(",")
wallet_dtype = {c: np.float32 for c in wallet_columns if c not in ("address", "Time step")}
wallet_dtype["address"] = "string"
wallet_dtype["Time step"] = np.int16
wallet_per_step_raw = pd.read_csv(WALLETS_DIR / "wallets_features.txt", dtype=wallet_dtype)
wallet_feat_cols = [c for c in wallet_per_step_raw.columns if c not in ("address", "Time step")]
wallet_per_step_raw[wallet_feat_cols] = wallet_per_step_raw[wallet_feat_cols].fillna(0).astype(np.float32)

# Per-row metadata (covers ALL rows, labelled and unlabelled)
all_addr_idx = wallet_per_step_raw["address"].map(wallet_addr_to_idx).astype(np.int64).values
all_t        = wallet_per_step_raw["Time step"].values.astype(np.int64)
all_X_raw    = wallet_per_step_raw[wallet_feat_cols].values.astype(np.float32)

# === Build labelled per-row table ===
merged = wallet_per_step_raw.merge(wallet_classes[["address", "label"]], on="address", how="left")
merged["label"] = merged["label"].fillna(-1).astype(np.int8)
labeled_mask_rowwise = (merged["label"] != -1).values
labeled_rows = merged[labeled_mask_rowwise].reset_index(drop=True)
labeled_rows["wallet_idx"] = labeled_rows["address"].map(wallet_addr_to_idx).astype(np.int64)
labeled_to_full_row = np.where(labeled_mask_rowwise)[0]   # maps labeled-row index → full-row index
print(f"labelled per-row table: {len(labeled_rows):,} rows  illicit_rate={labeled_rows['label'].mean():.4f}")

# === 70/30 per-WALLET stratified split (no leakage) ===
# Per-(wallet, time-step) features are 100% identical within a wallet, so a
# per-row random split puts every wallet on both sides and trivially leaks.
# The Elliptic++ paper itself does drop(["Time step"]).drop_duplicates() before
# splitting (Actors_Classification cell 17) — i.e. it splits at the wallet level.
# We do the same, with stratification + a fixed seed, then propagate the wallet
# assignment to every row of that wallet's per-step table.
unique_wallets = labeled_rows[["wallet_idx", "label"]].drop_duplicates(subset=["wallet_idx"]).reset_index(drop=True)
train_w, test_w = train_test_split(
    unique_wallets["wallet_idx"].values,
    test_size=TEST_SPLIT, random_state=RANDOM_SEED,
    stratify=unique_wallets["label"].values,
)
train_w_set = set(train_w.tolist()); test_w_set = set(test_w.tolist())
wallet_idx_per_row = labeled_rows["wallet_idx"].values
is_train_row = np.fromiter((w in train_w_set for w in wallet_idx_per_row), dtype=bool, count=len(wallet_idx_per_row))
is_test_row  = np.fromiter((w in test_w_set  for w in wallet_idx_per_row), dtype=bool, count=len(wallet_idx_per_row))
train_row_idx = np.where(is_train_row)[0]
test_row_idx  = np.where(is_test_row)[0]
y_per_row = labeled_rows["label"].values.astype(np.int64)
print(f"train wallets: {len(train_w):,}  test wallets: {len(test_w):,}")
print(f"train rows: {len(train_row_idx):,}  test rows: {len(test_row_idx):,}")

# === Fit scaler on LABELLED TRAINING rows only; standardise ALL per-step rows with it ===
X_train_for_scaler = all_X_raw[labeled_to_full_row[train_row_idx]]
wallet_scaler = StandardScaler().fit(X_train_for_scaler)
all_X_std = wallet_scaler.transform(all_X_raw).astype(np.float32)
X_per_row_std = all_X_std[labeled_to_full_row]   # standardised features for the labelled per-row table

addr_per_row = labeled_rows["wallet_idx"].values.astype(np.int64)
t_per_row    = labeled_rows["Time step"].values.astype(np.int64)

# === Per-wallet aggregated features (mean across ALL rows of that wallet) for static graph node features ===
df_all = pd.DataFrame(all_X_std, columns=wallet_feat_cols)
df_all["wallet_idx"] = all_addr_idx
wallet_node_feats_std = (df_all.groupby("wallet_idx")[wallet_feat_cols].mean()
                              .reindex(range(N_w)).fillna(0).values.astype(np.float32))
del df_all, merged

# === Per-step lookup arrays for the Hawkes rolling causal buffer ===
# step_rows_np[t] = positions in all_X_std for rows at time t
# step_wallets_np[t] = wallet indices for those rows
step_rows_np = [np.empty(0, dtype=np.int64) for _ in range(N_TIME_STEPS + 1)]
step_wallets_np = [np.empty(0, dtype=np.int64) for _ in range(N_TIME_STEPS + 1)]
for t in range(1, N_TIME_STEPS + 1):
    mask = all_t == t
    if mask.any():
        step_rows_np[t]    = np.where(mask)[0]
        step_wallets_np[t] = all_addr_idx[mask]

# === Tx node features ===
transactions = pd.read_csv(TRANSACTIONS_DIR / "txs_features.csv")
tx_feat_cols = [c for c in transactions.columns if c not in ("txId", "Time step")]
transactions[tx_feat_cols] = transactions[tx_feat_cols].fillna(0).astype(np.float32)
tx_id_to_idx = {tid: i for i, tid in enumerate(transactions["txId"].values)}
N_tx = len(transactions)
tx_features_array = transactions[tx_feat_cols].values
tx_time_steps_np  = transactions["Time step"].values.astype(np.int64)
tx_features_std = StandardScaler().fit(tx_features_array).transform(tx_features_array).astype(np.float32)

# === Edge files (actor heterogeneous graph only) ===
addr_addr_edges = pd.read_csv(WALLETS_DIR / "AddrAddr_edgelist.txt")
addr_tx_edges   = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
tx_addr_edges   = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")

hetero_graph = HeteroData()
hetero_graph["wallet"].x = torch.from_numpy(wallet_node_feats_std)
hetero_graph["tx"].x     = torch.from_numpy(tx_features_std)
hetero_graph["wallet", "addr_to_addr", "wallet"].edge_index = map_edges(addr_addr_edges, "input_address", "output_address", wallet_addr_to_idx, wallet_addr_to_idx)
hetero_graph["wallet", "addr_to_tx",   "tx"].edge_index     = map_edges(addr_tx_edges,   "input_address", "txId",            wallet_addr_to_idx, tx_id_to_idx)
hetero_graph["tx",     "tx_to_addr",   "wallet"].edge_index = map_edges(tx_addr_edges,   "txId",          "output_address",  tx_id_to_idx,       wallet_addr_to_idx)
hetero_graph = ToUndirected()(hetero_graph)

print("\nHetero graph (actor only):")
for ntype in hetero_graph.node_types:
    print(f"  {ntype:8s} nodes={hetero_graph[ntype].num_nodes:>9,}  features={hetero_graph[ntype].x.shape[1]}")
for et in hetero_graph.edge_types:
    print(f"  {str(et):60s} edges={hetero_graph[et].edge_index.shape[1]:>9,}")

# === Per-(wallet, t) event masks ===
event_label = torch.full((N_w, N_TIME_STEPS + 1), -1, dtype=torch.long)
event_label[addr_per_row, t_per_row] = torch.from_numpy(y_per_row)
event_train = torch.zeros((N_w, N_TIME_STEPS + 1), dtype=torch.bool)
event_train[addr_per_row[train_row_idx], t_per_row[train_row_idx]] = True
event_test  = torch.zeros((N_w, N_TIME_STEPS + 1), dtype=torch.bool)
event_test[addr_per_row[test_row_idx], t_per_row[test_row_idx]]   = True

# Per-step active masks
active_wallet = torch.zeros(N_TIME_STEPS + 1, N_w, dtype=torch.bool)
active_wallet[all_t, all_addr_idx] = True
active_tx = torch.zeros(N_TIME_STEPS + 1, N_tx, dtype=torch.bool)
active_tx[tx_time_steps_np, np.arange(N_tx)] = True

# Per-relation per-step edge active masks
active_by_type = {"wallet": active_wallet, "tx": active_tx}
edge_active_mask = {}
for etype in hetero_graph.edge_types:
    s_type, _, d_type = etype
    src_a = active_by_type[s_type]; dst_a = active_by_type[d_type]
    src, dst = hetero_graph[etype].edge_index[0], hetero_graph[etype].edge_index[1]
    edge_active_mask[etype] = src_a[:, src] & dst_a[:, dst]

del wallet_per_step_raw  # free the source DataFrame; numpy arrays remain

# Move tensors to device
hetero_graph = hetero_graph.to(DEVICE)
edge_active_mask = {k: v.to(DEVICE) for k, v in edge_active_mask.items()}
event_label = event_label.to(DEVICE)
event_train = event_train.to(DEVICE)
event_test  = event_test.to(DEVICE)
active_wallet_dev = active_wallet.to(DEVICE)

F_wallet = X_per_row_std.shape[1]
F_tx     = tx_features_std.shape[1]
print(f"\nF_wallet={F_wallet}  F_tx={F_tx}  N_w={N_w:,}  N_tx={N_tx:,}")

labelled per-row table: 367,472 rows  illicit_rate=0.0778
train rows: 257,230  test rows: 110,242

Hetero graph (actor only):
  wallet   nodes=  822,942  features=55
  tx       nodes=  203,769  features=182
  ('wallet', 'addr_to_addr', 'wallet')                         edges=5,553,655
  ('wallet', 'addr_to_tx', 'tx')                               edges=  477,117
  ('tx', 'tx_to_addr', 'wallet')                               edges=  837,124
  ('tx', 'rev_addr_to_tx', 'wallet')                           edges=  477,117
  ('wallet', 'rev_tx_to_addr', 'tx')                           edges=  837,124

F_wallet=55  F_tx=182  N_w=822,942  N_tx=203,769


## 2. Tabular baselines: Random Forest and Logistic Regression

Per-row 70/30 stratified random split. Class-balanced weights. These numbers should land near the original Elliptic++ paper's RF (~0.83 illicit F1).

In [5]:
results = {}

def report(y_true, y_pred, y_proba, name):
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    auc = roc_auc_score(y_true, y_proba) if y_proba is not None else float("nan")
    print(f"[{name}] illicit F1={f1:.4f}  AUC={auc:.4f}")
    print(classification_report(y_true, y_pred, target_names=["licit", "illicit"], digits=4, zero_division=0))
    results[name] = {"f1": f1, "auc": auc}
    return f1, auc

y_test = y_per_row[test_row_idx]
X_train_std = X_per_row_std[train_row_idx]
y_train = y_per_row[train_row_idx]
X_test_std  = X_per_row_std[test_row_idx]

print("=== Random Forest ===")
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                            n_jobs=-1, random_state=RANDOM_SEED)
rf.fit(X_train_std, y_train)
rf_pred  = rf.predict(X_test_std)
rf_proba = rf.predict_proba(X_test_std)[:, 1]
report(y_test, rf_pred, rf_proba, "RF")

print("\n=== Logistic Regression ===")
lr = LogisticRegression(class_weight="balanced", max_iter=2000, solver="saga",
                        n_jobs=-1, random_state=RANDOM_SEED)
lr.fit(X_train_std, y_train)
lr_pred  = lr.predict(X_test_std)
lr_proba = lr.predict_proba(X_test_std)[:, 1]
report(y_test, lr_pred, lr_proba, "LR")

=== Random Forest ===
[RF] illicit F1=0.9539  AUC=0.9978
              precision    recall  f1-score   support

       licit     0.9936    0.9989    0.9962    101662
     illicit     0.9867    0.9232    0.9539      8580

    accuracy                         0.9931    110242
   macro avg     0.9901    0.9611    0.9751    110242
weighted avg     0.9930    0.9931    0.9929    110242


=== Logistic Regression ===
[LR] illicit F1=0.3916  AUC=0.8461
              precision    recall  f1-score   support

       licit     0.9757    0.8219    0.8922    101662
     illicit     0.2641    0.7571    0.3916      8580

    accuracy                         0.8169    110242
   macro avg     0.6199    0.7895    0.6419    110242
weighted avg     0.9203    0.8169    0.8533    110242



(0.3915729829108774, 0.8461241720874131)

## 3. Heterogeneous GNN embedder (configurable conv type and depth)

`HeteroEmbedder` runs over the actor heterogeneous graph (wallet ↔ wallet via `AddrAddr`, wallet ↔ tx via `AddrTx` and `TxAddr`, plus reverses) and produces a per-wallet hidden state.

- `conv_type="sage"` — `SAGEConv` for every relation (handles same-type and bipartite).
- `conv_type="gcn"` — `GCNConv` for the same-type wallet↔wallet relations, `SAGEConv` for the bipartite wallet↔tx relations (vanilla GCN is undefined on bipartite). Mixed by necessity.
- `num_layers` — number of `HeteroConv` layers stacked.

We train it supervised on per-wallet labels (transductive — all 265K labelled wallets contribute, since the heterogeneous graph is shared between train and test rows). The penultimate hidden state is the embedding fed to the Temporal Transformer below.

In [11]:
class HeteroEmbedder(nn.Module):
    """Heterogeneous GNN with configurable conv_type and num_layers.
    Returns per-wallet hidden states and (optionally) classification logits."""
    def __init__(self, edge_types, wallet_dim, tx_dim,
                 conv_type="sage", num_layers=3, hidden_dim=128, dropout=0.4):
        super().__init__()
        if conv_type not in ("gcn", "sage"):
            raise ValueError(f"conv_type must be 'gcn' or 'sage', got {conv_type}")
        self.conv_type = conv_type
        self.num_layers = num_layers
        self.input_proj = nn.ModuleDict({
            "wallet": nn.Linear(wallet_dim, hidden_dim),
            "tx":     nn.Linear(tx_dim,     hidden_dim),
        })
        def build_conv(rel):
            src_t, _, dst_t = rel
            if src_t == dst_t and conv_type == "gcn":
                return GCNConv(hidden_dim, hidden_dim, add_self_loops=False)
            return SAGEConv((-1, -1), hidden_dim)
        self.layers = nn.ModuleList([
            HeteroConv({rel: build_conv(rel) for rel in edge_types}, aggr="sum")
            for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([
            nn.ModuleDict({nt: nn.LayerNorm(hidden_dim) for nt in ["wallet", "tx"]})
            for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict, return_embedding=False):
        h_dict = {nt: self.input_proj[nt](x) for nt, x in x_dict.items()}
        for layer, norm in zip(self.layers, self.norms):
            h_dict = layer(h_dict, edge_index_dict)
            h_dict = {nt: F.relu(norm[nt](v)) for nt, v in h_dict.items()}
            h_dict = {nt: F.dropout(v, p=self.dropout, training=self.training)
                      for nt, v in h_dict.items()}
        wallet_h = h_dict["wallet"]
        logits = self.classifier(wallet_h)
        return (logits, wallet_h) if return_embedding else logits

In [12]:
def train_embedder(embedder, hetero, train_wallet_mask, wallet_labels,
                   epochs=30, lr=1e-3, weight_decay=5e-4, name="embedder"):
    embedder = embedder.to(DEVICE)
    optim = torch.optim.Adam(embedder.parameters(), lr=lr, weight_decay=weight_decay)
    train_idx = torch.tensor(np.where(train_wallet_mask)[0], dtype=torch.long, device=DEVICE)
    labels = torch.tensor(wallet_labels, dtype=torch.long, device=DEVICE)
    n_pos = int((labels[train_idx] == 1).sum())
    n_neg = int((labels[train_idx] == 0).sum())
    cw = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float, device=DEVICE)
    for epoch in range(1, epochs + 1):
        embedder.train(); optim.zero_grad()
        logits = embedder(hetero.x_dict, hetero.edge_index_dict)
        loss = F.cross_entropy(logits[train_idx], labels[train_idx], weight=cw)
        loss.backward(); optim.step()
        if epoch % 5 == 0 or epoch == epochs:
            print(f"  [{name}] epoch {epoch:2d}  loss={loss.item():.4f}")
    return embedder

# A wallet is in the embedder's training set if it has at least one row in the per-row training split.
train_wallets_set = set(addr_per_row[train_row_idx].tolist())
train_wallet_mask = np.zeros(N_w, dtype=bool)
train_wallet_mask[list(train_wallets_set)] = True
train_wallet_mask &= (wallet_label_per_addr != -1)
print(f"Embedder training wallets: {train_wallet_mask.sum():,} (labelled wallets with ≥1 train row)")

extracted_embeddings = {}
embedder_types_to_run = ["gcn", "sage"] if RUN_BOTH_EMBEDDERS else [EMBEDDER_CONV_TYPE]
for conv_type in embedder_types_to_run:
    print(f"\n=== {conv_type.upper()} embedder ({EMBEDDER_NUM_LAYERS} layers, hidden={EMBEDDER_HIDDEN_DIM}) ===")
    emb = HeteroEmbedder(
        edge_types=hetero_graph.edge_types,
        wallet_dim=F_wallet, tx_dim=F_tx,
        conv_type=conv_type, num_layers=EMBEDDER_NUM_LAYERS,
        hidden_dim=EMBEDDER_HIDDEN_DIM, dropout=EMBEDDER_DROPOUT,
    )
    trained = train_embedder(
        emb, hetero_graph, train_wallet_mask,
        wallet_label_per_addr.astype(np.int64),
        epochs=EMBEDDER_EPOCHS, lr=EMBEDDER_LR, name=conv_type,
    )
    trained.eval()
    with torch.no_grad():
        _, emb_t = trained(hetero_graph.x_dict, hetero_graph.edge_index_dict, return_embedding=True)
    extracted_embeddings[conv_type] = emb_t.cpu().numpy()
    del trained, emb
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

for k, v in extracted_embeddings.items():
    print(f"{k.upper()} embeddings: shape={v.shape}")

Embedder training wallets: 197,567 (labelled wallets with ≥1 train row)

=== GCN embedder (3 layers, hidden=128) ===
  [gcn] epoch  5  loss=0.3529
  [gcn] epoch 10  loss=0.2437
  [gcn] epoch 15  loss=0.2156
  [gcn] epoch 20  loss=0.1876
  [gcn] epoch 25  loss=0.1660
  [gcn] epoch 30  loss=0.1504
  [gcn] epoch 35  loss=0.1331
  [gcn] epoch 40  loss=0.1197
  [gcn] epoch 45  loss=0.1080
  [gcn] epoch 50  loss=0.0976

=== SAGE embedder (3 layers, hidden=128) ===
  [sage] epoch  5  loss=0.3362
  [sage] epoch 10  loss=0.2470
  [sage] epoch 15  loss=0.2169
  [sage] epoch 20  loss=0.1921
  [sage] epoch 25  loss=0.1666
  [sage] epoch 30  loss=0.1524
  [sage] epoch 35  loss=0.1382
  [sage] epoch 40  loss=0.1248
  [sage] epoch 45  loss=0.1132
  [sage] epoch 50  loss=0.1026
GCN embeddings: shape=(822942, 128)
SAGE embeddings: shape=(822942, 128)


## 4. Temporal Transformer with GNN embedding

For each wallet, build a sequence of its labelled per-time-step rows (sorted by `Time step`). Each token concatenates the per-step features with the wallet's static GNN embedding (broadcast across time), is projected to `d_model`, summed with a learned positional embedding keyed by absolute `Time step`, and passed through a Transformer encoder. Per-token logits classify each `(wallet, time-step)` row.

Per-row 70/30 split: each token's loss/eval is masked by whether its source row is in `train_row_idx` / `test_row_idx`. A wallet's sequence may contain a mix of train and test tokens; only train tokens contribute to the loss.

We sweep both GCN and SAGE embeddings (when `RUN_BOTH_EMBEDDERS=True`).

In [23]:
# === Build per-wallet sequences ===
labeled_rows["orig_idx"] = np.arange(len(labeled_rows))
sorted_rows = labeled_rows.sort_values(["wallet_idx", "Time step"]).reset_index(drop=True)
is_train = np.zeros(len(labeled_rows), dtype=bool); is_train[train_row_idx] = True
is_test  = np.zeros(len(labeled_rows), dtype=bool); is_test[test_row_idx]  = True

# Determine T_MAX (longest wallet sequence)
T_MAX = min(int(sorted_rows.groupby("wallet_idx").size().max()),16)
print(f"T_MAX (longest wallet sequence): {T_MAX}")

groups = sorted_rows.groupby("wallet_idx", sort=False)
n_seqs = len(groups)
seq_x = np.zeros((n_seqs, T_MAX, F_wallet), dtype=np.float32)
seq_t = np.zeros((n_seqs, T_MAX), dtype=np.int64)
seq_pad_mask  = np.ones((n_seqs, T_MAX), dtype=bool)   # True = padding
seq_train_mask = np.zeros((n_seqs, T_MAX), dtype=bool)
seq_test_mask  = np.zeros((n_seqs, T_MAX), dtype=bool)
seq_labels = np.zeros((n_seqs, T_MAX), dtype=np.int64)
seq_wallet_idx = np.zeros(n_seqs, dtype=np.int64)

for i, (wallet, group) in enumerate(groups):
    if len(group) > T_MAX:
          group = group.iloc[-T_MAX:]
    n = len(group)
    orig_indices = group["orig_idx"].values
    seq_x[i, :n] = X_per_row_std[orig_indices]
    seq_t[i, :n] = group["Time step"].values
    seq_pad_mask[i, :n]   = False
    seq_train_mask[i, :n] = is_train[orig_indices]
    seq_test_mask[i, :n]  = is_test[orig_indices]
    seq_labels[i, :n]     = group["label"].values
    seq_wallet_idx[i] = wallet

print(f"Sequences: {n_seqs:,} wallets × T_MAX={T_MAX}")
print(f"  train tokens: {seq_train_mask.sum():,}  test tokens: {seq_test_mask.sum():,}  pad tokens: {seq_pad_mask.sum():,}")

X_seq_t = torch.from_numpy(seq_x)
T_seq_t = torch.from_numpy(seq_t)
P_seq_t = torch.from_numpy(seq_pad_mask)
TR_seq_t = torch.from_numpy(seq_train_mask)
TE_seq_t = torch.from_numpy(seq_test_mask)
Y_seq_t = torch.from_numpy(seq_labels)
W_seq_t = torch.from_numpy(seq_wallet_idx)

T_MAX (longest wallet sequence): 16
Sequences: 265,354 wallets × T_MAX=16
  train tokens: 244,100  test tokens: 104,616  pad tokens: 3,896,948


In [24]:
class TemporalTransformerWithEmbedding(nn.Module):
    def __init__(self, seq_dim, emb_dim, d_model=128, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.token_proj = nn.Linear(seq_dim + emb_dim, d_model)
        self.position = nn.Embedding(N_TIME_STEPS + 1, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model, n_heads, dim_feedforward=4 * d_model,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.classifier = nn.Linear(d_model, 2)

    def forward(self, x, t, pad_mask, emb):
        emb_b = emb.unsqueeze(1).expand(-1, x.size(1), -1)
        token = torch.cat([x, emb_b], dim=-1)
        h = self.token_proj(token) + self.position(t)
        h = self.encoder(h, src_key_padding_mask=pad_mask)
        return self.classifier(h)  # [B, T, 2]

def train_temporal_transformer(emb_array, name):
    """emb_array: [N_w, F_emb] numpy array of per-wallet embeddings."""
    F_emb = emb_array.shape[1]
    print(f"\n=== Temporal Transformer — embedding={name}  emb_dim={F_emb} ===")
    # Per-wallet embedding tensor aligned with seq_wallet_idx
    seq_emb = torch.from_numpy(emb_array[seq_wallet_idx])  # [n_seqs, F_emb]
    n_train = int(TR_seq_t.sum())
    n_pos = int(((Y_seq_t == 1) & TR_seq_t).sum())
    n_neg = int(((Y_seq_t == 0) & TR_seq_t).sum())
    cw = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float, device=DEVICE)
    print(f"train tokens: {n_train:,}  illicit_weight={cw[1]:.2f}")

    model = TemporalTransformerWithEmbedding(
        seq_dim=F_wallet, emb_dim=F_emb,
        d_model=TEMPORAL_D_MODEL, n_heads=TEMPORAL_N_HEADS,
        n_layers=TEMPORAL_N_LAYERS, dropout=TEMPORAL_DROPOUT,
    ).to(DEVICE)
    optim = torch.optim.Adam(model.parameters(), lr=TEMPORAL_LR, weight_decay=5e-4)

    loader = DataLoader(
        TensorDataset(X_seq_t, T_seq_t, P_seq_t, seq_emb, Y_seq_t, TR_seq_t, TE_seq_t),
        batch_size=TEMPORAL_BATCH_SIZE, shuffle=True,
    )
    eval_loader = DataLoader(
        TensorDataset(X_seq_t, T_seq_t, P_seq_t, seq_emb, Y_seq_t, TR_seq_t, TE_seq_t),
        batch_size=TEMPORAL_BATCH_SIZE * 2, shuffle=False,
    )

    for epoch in range(1, TEMPORAL_EPOCHS + 1):
        model.train(); running, seen = 0.0, 0
        for x, t, pm, em, y, tr, te in loader:
            x, t, pm = x.to(DEVICE), t.to(DEVICE), pm.to(DEVICE)
            em, y, tr = em.to(DEVICE), y.to(DEVICE), tr.to(DEVICE)
            optim.zero_grad()
            logits = model(x, t, pm, em)  # [B, T, 2]
            ce = F.cross_entropy(
                logits.reshape(-1, 2), y.reshape(-1),
                weight=cw, reduction="none",
            ).reshape(logits.shape[:2])
            loss = (ce * tr.float()).sum() / tr.float().sum().clamp(min=1)
            loss.backward(); optim.step()
            running += loss.item() * int(tr.sum()); seen += int(tr.sum())
        if epoch % 5 == 0 or epoch == TEMPORAL_EPOCHS:
            print(f"  epoch {epoch:2d}  train loss={running/max(seen,1):.4f}")

    # Eval — collect per-token predictions on test rows
    model.eval()
    y_all, p_all, prob_all = [], [], []
    with torch.no_grad():
        for x, t, pm, em, y, tr, te in eval_loader:
            x, t, pm, em = x.to(DEVICE), t.to(DEVICE), pm.to(DEVICE), em.to(DEVICE)
            logits = model(x, t, pm, em).softmax(-1)
            te_b = te.to(DEVICE)
            mask_flat = te_b.reshape(-1)
            if not mask_flat.any():
                continue
            probs = logits.reshape(-1, 2)[mask_flat, 1]
            preds = logits.reshape(-1, 2)[mask_flat].argmax(-1)
            ys    = y.to(DEVICE).reshape(-1)[mask_flat]
            y_all.append(ys.cpu().numpy())
            p_all.append(preds.cpu().numpy())
            prob_all.append(probs.cpu().numpy())
    y_arr = np.concatenate(y_all); p_arr = np.concatenate(p_all); prob_arr = np.concatenate(prob_all)
    report(y_arr, p_arr, prob_arr, f"Transformer+{name.upper()}")
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

for conv_type in embedder_types_to_run:
    train_temporal_transformer(extracted_embeddings[conv_type], conv_type)


=== Temporal Transformer — embedding=gcn  emb_dim=128 ===
train tokens: 244,100  illicit_weight=11.80
  epoch  5  train loss=0.1320
  epoch 10  train loss=0.1198
  epoch 15  train loss=0.1161
  epoch 20  train loss=0.1122
  epoch 25  train loss=0.1096
  epoch 30  train loss=0.1074
  epoch 35  train loss=0.1057
  epoch 40  train loss=0.1022
  epoch 45  train loss=0.0999
  epoch 50  train loss=0.1086
[Transformer+GCN] illicit F1=0.8565  AUC=0.9978
              precision    recall  f1-score   support

       licit     0.9983    0.9738    0.9859     96430
     illicit     0.7603    0.9805    0.8565      8186

    accuracy                         0.9743    104616
   macro avg     0.8793    0.9771    0.9212    104616
weighted avg     0.9797    0.9743    0.9758    104616


=== Temporal Transformer — embedding=sage  emb_dim=128 ===
train tokens: 244,100  illicit_weight=11.80
  epoch  5  train loss=0.1380
  epoch 10  train loss=0.1289
  epoch 15  train loss=0.1241
  epoch 20  train loss=0.122

## 5. Graph Attention Hawkes (improved)

Same dual-attention CT-LSTM architecture as `graph_hawkes.ipynb`, with the upgrades discussed:

- **Full labelled population** (~265K wallets get CT-LSTM state, vs 54K subsample before). Unlabelled wallets still participate in graph attention as context.
- **`hidden_dim=128`** (vs 64).
- **Focal loss** (`γ=2`) with class weight, replacing weighted CE.
- **Threshold calibration** — carve a small validation slice from the 70% train rows, sweep thresholds on it, apply the F1-maximising threshold to test.
- **Mixed precision** (bf16) when on CUDA.
- **Causal feature buffer** for wallet GNN inputs (no leakage — the bug we fixed in `graph_hawkes.ipynb`).
- **Per-row 70/30 split applied as event-level masks**: only events in `train_row_idx` contribute to loss; only events in `test_row_idx` enter the test metric.

In [32]:
class CTLSTMCell(nn.Module):
    """Mei & Eisner 2017 7-gate continuous-time LSTM cell."""
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.input_to_gates  = nn.Linear(input_dim, 7 * hidden_dim, bias=False)
        self.hidden_to_gates = nn.Linear(hidden_dim, 7 * hidden_dim)
    @staticmethod
    def decay(c, c_bar, delta, dt):
        if dt.ndim == 1: dt = dt.unsqueeze(-1)
        return c_bar + (c - c_bar) * torch.exp(-delta * dt)
    def update(self, x, c_decayed, c_bar_prev, h_decayed):
        gates = self.input_to_gates(x) + self.hidden_to_gates(h_decayed)
        i, f, z, o, i_bar, f_bar, delta = gates.chunk(7, dim=-1)
        i, f, o, i_bar, f_bar = (torch.sigmoid(g) for g in (i, f, o, i_bar, f_bar))
        z = 2 * torch.sigmoid(z) - 1
        delta = F.softplus(delta)
        c     = f * c_decayed + i * z
        c_bar = f_bar * c_bar_prev + i_bar * z
        h     = o * torch.tanh(c)
        return c, c_bar, delta, o, h

class TemporalSelfAttention(nn.Module):
    """Causal multi-head attention with additive time-decay bias and ring-buffer cache."""
    def __init__(self, hidden_dim, n_heads=4, beta_decay_init=0.2):
        super().__init__()
        assert hidden_dim % n_heads == 0
        self.hidden_dim, self.n_heads = hidden_dim, n_heads
        self.head_dim = hidden_dim // n_heads
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        raw = float(np.log(np.exp(beta_decay_init) - 1))
        self.beta_decay_raw = nn.Parameter(torch.full((n_heads,), raw))
    def forward(self, query, cache_h, cache_t, cur_t, fill):
        B, W, H = cache_h.shape
        if B == 0: return query.new_zeros(0, H)
        Q = self.q_proj(query).view(B, self.n_heads, self.head_dim)
        K = self.k_proj(cache_h).view(B, W, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(cache_h).view(B, W, self.n_heads, self.head_dim).transpose(1, 2)
        scores = torch.einsum("bhd,bhwd->bhw", Q, K) / (self.head_dim ** 0.5)
        beta = F.softplus(self.beta_decay_raw)
        time_diff = (cur_t.unsqueeze(-1) - cache_t).float()
        scores = scores + (-beta.view(1, -1, 1)) * time_diff.unsqueeze(1)
        positions = torch.arange(W, device=fill.device).unsqueeze(0)
        pad = positions >= fill.unsqueeze(-1)
        scores = scores.masked_fill(pad.unsqueeze(1), float("-inf"))
        attn = torch.nan_to_num(F.softmax(scores, dim=-1), nan=0.0)
        z = torch.einsum("bhw,bhwd->bhd", attn, V).reshape(B, H)
        return self.out_proj(z)

class HeteroGraphAttention(nn.Module):
    """GAT on same-type relations, SAGE on cross-type relations; sum across relations."""
    def __init__(self, edge_types, hidden_dim, gat_heads=2, dropout=0.2):
        super().__init__()
        per_head = hidden_dim // gat_heads
        convs = {}
        for rel in edge_types:
            same = (rel[0] == rel[2])
            if same:
                convs[rel] = GATConv((-1, -1), per_head, heads=gat_heads, dropout=dropout, add_self_loops=False)
            else:
                convs[rel] = SAGEConv((-1, -1), hidden_dim)
        self.conv = HeteroConv(convs, aggr="sum")
        self.norms = nn.ModuleDict({"wallet": nn.LayerNorm(hidden_dim),
                                    "tx":     nn.LayerNorm(hidden_dim)})
    def forward(self, x_dict, edge_index_dict):
        out = self.conv(x_dict, edge_index_dict)
        return {k: F.relu(self.norms[k](v)) for k, v in out.items()}

In [33]:
class GraphAttentiveNeuralHawkes(nn.Module):
    def __init__(self, wallet_dim, tx_dim, edge_types,
                 hidden_dim=128, window=8, n_attn_heads=4, gat_heads=2,
                 beta_decay_init=0.2, fuse_mode="gated", dropout=0.2):
        super().__init__()
        self.hidden_dim, self.window, self.fuse_mode = hidden_dim, window, fuse_mode
        self.wallet_proj = nn.Linear(wallet_dim, hidden_dim)
        self.tx_proj = nn.Linear(tx_dim, hidden_dim)
        self.graph_attn = HeteroGraphAttention(edge_types, hidden_dim, gat_heads=gat_heads, dropout=dropout)
        self.temporal_attn = TemporalSelfAttention(hidden_dim, n_heads=n_attn_heads, beta_decay_init=beta_decay_init)
        if fuse_mode == "concat":
            self.fuse = nn.Linear(2 * hidden_dim, hidden_dim)
        elif fuse_mode == "gated":
            self.fuse_gate = nn.Linear(2 * hidden_dim, hidden_dim)
            self.fuse_t = nn.Linear(hidden_dim, hidden_dim)
            self.fuse_g = nn.Linear(hidden_dim, hidden_dim)
        elif fuse_mode == "sum":
            self.fuse_t = nn.Linear(hidden_dim, hidden_dim)
            self.fuse_g = nn.Linear(hidden_dim, hidden_dim)
        else:
            raise ValueError(fuse_mode)
        self.ct_lstm = CTLSTMCell(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2)

    def fuse_attentions(self, z_time, z_graph):
        if self.fuse_mode == "concat":
            return self.fuse(torch.cat([z_time, z_graph], dim=-1))
        if self.fuse_mode == "gated":
            g = torch.sigmoid(self.fuse_gate(torch.cat([z_time, z_graph], dim=-1)))
            return g * self.fuse_t(z_time) + (1 - g) * self.fuse_g(z_graph)
        return self.fuse_t(z_time) + self.fuse_g(z_graph)

    def init_state(self, n_subset, device):
        H, W = self.hidden_dim, self.window
        return {
            "c":       torch.zeros(n_subset, H, device=device),
            "c_bar":   torch.zeros(n_subset, H, device=device),
            "delta":   torch.full((n_subset, H), 0.1, device=device),
            "o":       torch.zeros(n_subset, H, device=device),
            "last_t":  torch.zeros(n_subset, dtype=torch.long, device=device),
            "cache_h": torch.zeros(n_subset, W, H, device=device),
            "cache_t": torch.zeros(n_subset, W, dtype=torch.long, device=device),
            "fill":    torch.zeros(n_subset, dtype=torch.long, device=device),
        }

    @staticmethod
    def detach_state(state):
        return {k: (v.detach() if torch.is_tensor(v) else v) for k, v in state.items()}

    def encode_graph(self, x_wallet, x_tx, edge_index_dict_t):
        x_dict = {"wallet": self.wallet_proj(x_wallet), "tx": self.tx_proj(x_tx)}
        return self.graph_attn(x_dict, edge_index_dict_t)

    def step(self, t, z_graph_wallet_full, active_subidx, subset_to_global, state):
        if active_subidx.numel() == 0:
            return None, state
        H, W = self.hidden_dim, self.window
        device = z_graph_wallet_full.device
        active_global = subset_to_global[active_subidx]
        z_graph = z_graph_wallet_full[active_global]
        c     = state["c"][active_subidx]
        c_bar = state["c_bar"][active_subidx]
        delta = state["delta"][active_subidx]
        o     = state["o"][active_subidx]
        last_t = state["last_t"][active_subidx]
        dt = (t - last_t).clamp(min=0).float()
        c_dec = self.ct_lstm.decay(c, c_bar, delta, dt)
        h_dec = o * torch.tanh(c_dec)
        cache_h_act = state["cache_h"][active_subidx]
        cache_t_act = state["cache_t"][active_subidx]
        fill_act    = state["fill"][active_subidx]
        cur_t = torch.full((active_subidx.numel(),), t, dtype=torch.long, device=device)
        z_time = self.temporal_attn(h_dec, cache_h_act, cache_t_act, cur_t, fill_act)
        x_fused = self.fuse_attentions(z_time, z_graph)
        c_new, c_bar_new, delta_new, o_new, h_new = self.ct_lstm.update(x_fused, c_dec, c_bar, h_dec)

        # Cast gate outputs back to state dtype — under torch.autocast, the gate
        # computations produce bfloat16 but state tensors are float32, so index_copy
        # would reject a dtype mismatch otherwise. The classifier head (which feeds
        # into the loss) keeps the autocast dtype; the loss-side .float() cast in
        # the training loop handles the cross-entropy correctly.
        state_fp = state["c"].dtype
        c_new      = c_new.to(state_fp)
        c_bar_new  = c_bar_new.to(state_fp)
        delta_new  = delta_new.to(state_fp)
        o_new      = o_new.to(state_fp)
        h_new_state = h_new.to(state["cache_h"].dtype)

        new_state = dict(state)
        new_state["c"]      = state["c"].index_copy(0, active_subidx, c_new)
        new_state["c_bar"]  = state["c_bar"].index_copy(0, active_subidx, c_bar_new)
        new_state["delta"]  = state["delta"].index_copy(0, active_subidx, delta_new)
        new_state["o"]      = state["o"].index_copy(0, active_subidx, o_new)
        new_state["last_t"] = state["last_t"].index_copy(0, active_subidx, torch.full_like(active_subidx, t))
        slot_ring = (fill_act % W)
        flat_idx = active_subidx * W + slot_ring
        cache_h_flat = state["cache_h"].view(-1, H).index_copy(0, flat_idx, h_new_state)
        new_state["cache_h"] = cache_h_flat.view(-1, W, H)
        cache_t_flat = state["cache_t"].view(-1).index_copy(0, flat_idx, torch.full_like(flat_idx, t))
        new_state["cache_t"] = cache_t_flat.view(-1, W)
        new_state["fill"]    = state["fill"].index_copy(0, active_subidx, (fill_act + 1).clamp(max=W))
        logits = self.classifier(h_new)
        return logits, new_state

def focal_loss(logits, targets, gamma=2.0, weight=None):
    log_probs = F.log_softmax(logits, dim=-1)
    log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
    pt = log_pt.exp()
    fl = -((1 - pt) ** gamma) * log_pt
    if weight is not None:
        fl = weight[targets] * fl
    return fl.sum()  # caller normalises by event count

In [36]:
# === Subset = full labelled population ===
subset_global_np = np.where(wallet_label_per_addr != -1)[0]
N_subset = len(subset_global_np)
subset_to_global = torch.tensor(subset_global_np, dtype=torch.long, device=DEVICE)
global_to_subset = torch.full((N_w,), -1, dtype=torch.long, device=DEVICE)
global_to_subset[subset_to_global] = torch.arange(N_subset, device=DEVICE)
print(f"Hawkes subset size (all labelled wallets): {N_subset:,}")

# Active subset wallets per time step
active_subidx_per_step = []
for t in range(N_TIME_STEPS + 1):
    a_global = active_wallet_dev[t].nonzero(as_tuple=False).squeeze(-1)
    sub = global_to_subset[a_global]
    active_subidx_per_step.append(sub[sub >= 0])

# Per-step rolling causal feature buffer source — uses ACTUAL per-step features, not per-wallet means
step_wallets = [torch.empty(0, dtype=torch.long, device=DEVICE) for _ in range(N_TIME_STEPS + 1)]
step_features = [torch.empty(0, F_wallet, dtype=torch.float32, device=DEVICE) for _ in range(N_TIME_STEPS + 1)]
for t in range(1, N_TIME_STEPS + 1):
    if step_wallets_np[t].size > 0:
        step_wallets[t]  = torch.tensor(step_wallets_np[t], dtype=torch.long, device=DEVICE)
        step_features[t] = torch.tensor(all_X_std[step_rows_np[t]], device=DEVICE)

# === Carve a small validation slice from train rows for threshold calibration + best-epoch selection ===
rng = np.random.default_rng(RANDOM_SEED + 1)
val_size = int(0.1 * len(labeled_rows))
shuffled = rng.permutation(train_row_idx)
val_for_thresh_idx = shuffled[:val_size]
fit_idx = shuffled[val_size:]
event_val = torch.zeros((N_w, N_TIME_STEPS + 1), dtype=torch.bool, device=DEVICE)
event_val[addr_per_row[val_for_thresh_idx], t_per_row[val_for_thresh_idx]] = True
event_train_only = event_train.clone()
event_train_only[addr_per_row[val_for_thresh_idx], t_per_row[val_for_thresh_idx]] = False
print(f"Hawkes splits: train_fit={len(fit_idx):,}  val_for_threshold={len(val_for_thresh_idx):,}  test={len(test_row_idx):,}")

# === Class weight (computed on training-fit events only) ===
n_pos_train = int((event_label[event_train_only] == 1).sum())
n_neg_train = int((event_label[event_train_only] == 0).sum())
class_weight = torch.tensor([1.0, n_neg_train / max(n_pos_train, 1)], dtype=torch.float, device=DEVICE)
print(f"Train events: pos={n_pos_train:,}  neg={n_neg_train:,}  illicit_weight={class_weight[1]:.2f}")

# === Training loop ===
model = GraphAttentiveNeuralHawkes(
    wallet_dim=F_wallet, tx_dim=F_tx, edge_types=hetero_graph.edge_types,
    hidden_dim=HAWKES_HIDDEN_DIM, window=HAWKES_WINDOW,
    n_attn_heads=HAWKES_N_ATTN_HEADS, gat_heads=HAWKES_GAT_HEADS,
    beta_decay_init=HAWKES_BETA_DECAY_INIT, fuse_mode=HAWKES_FUSE_MODE,
    dropout=HAWKES_DROPOUT,
).to(DEVICE)
optim_h = torch.optim.Adam(model.parameters(), lr=HAWKES_LR, weight_decay=HAWKES_WEIGHT_DECAY)
amp_enabled = USE_AMP and DEVICE.type == "cuda"
amp_dtype = torch.bfloat16 if amp_enabled else torch.float32

def hawkes_run_epoch(train_mode):
    if train_mode: model.train()
    else:          model.eval()
    state = model.init_state(N_subset, DEVICE)
    wallet_buffer = torch.zeros(N_w, F_wallet, device=DEVICE)
    seg_loss, seg_n = None, 0
    epoch_loss, epoch_n = 0.0, 0
    val_y, val_p, test_y, test_p = [], [], [], []
    for t in range(1, N_TIME_STEPS + 1):
        if step_wallets[t].numel() > 0:
            wallet_buffer = wallet_buffer.index_copy(0, step_wallets[t], step_features[t])
        active_sub = active_subidx_per_step[t]
        if active_sub.numel() > 0:
            edge_index_t = {rel: hetero_graph[rel].edge_index[:, edge_active_mask[rel][t]]
                            for rel in hetero_graph.edge_types}
            ctx = torch.enable_grad() if train_mode else torch.no_grad()
            with ctx, torch.autocast(device_type=DEVICE.type, dtype=amp_dtype, enabled=amp_enabled):
                z_graph = model.encode_graph(wallet_buffer, hetero_graph["tx"].x, edge_index_t)
                logits, state = model.step(t, z_graph["wallet"], active_sub, subset_to_global, state)
            if logits is not None:
                active_global = subset_to_global[active_sub]
                tr_e = event_train_only[active_global, t]
                va_e = event_val[active_global, t]
                te_e = event_test[active_global, t]
                lab_e = event_label[active_global, t].long()
                if train_mode and tr_e.any():
                    sub_logits = logits.float()[tr_e]
                    sub_lab = lab_e[tr_e]
                    s_loss = focal_loss(sub_logits, sub_lab, gamma=HAWKES_FOCAL_GAMMA, weight=class_weight)
                    seg_loss = s_loss if seg_loss is None else seg_loss + s_loss
                    seg_n += int(tr_e.sum())
                if not train_mode:
                    if va_e.any():
                        prob = logits.float().softmax(-1)[:, 1][va_e]
                        val_y.append(lab_e[va_e].cpu().numpy()); val_p.append(prob.cpu().numpy())
                    if te_e.any():
                        prob = logits.float().softmax(-1)[:, 1][te_e]
                        test_y.append(lab_e[te_e].cpu().numpy()); test_p.append(prob.cpu().numpy())
        if train_mode and (t % HAWKES_TBPTT_K == 0 or t == N_TIME_STEPS):
            if seg_loss is not None and seg_n > 0:
                optim_h.zero_grad()
                (seg_loss / seg_n).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optim_h.step()
                epoch_loss += float(seg_loss.detach()); epoch_n += seg_n
            seg_loss, seg_n = None, 0
            state = model.detach_state(state)
            wallet_buffer = wallet_buffer.detach()
    if train_mode:
        return epoch_loss / max(epoch_n, 1)
    return ((np.concatenate(val_y) if val_y else np.array([])),
            (np.concatenate(val_p) if val_p else np.array([])),
            (np.concatenate(test_y) if test_y else np.array([])),
            (np.concatenate(test_p) if test_p else np.array([])))

def best_threshold_f1(y, p):
    """Return (best_threshold, best_f1) by sweeping the precision-recall curve on (y, p)."""
    if not len(y) or len(np.unique(y)) < 2:
        return 0.5, 0.0
    prec, rec, thr = precision_recall_curve(y, p)
    f1s = 2 * prec * rec / (prec + rec + 1e-10)
    best = int(np.nanargmax(f1s[:-1]))
    return float(thr[best]), float(f1s[best])

# === Best-epoch checkpointing on val oracle F1 ===
best_val_oracle_f1 = -1.0
best_state_dict = None
best_val_y, best_val_prob = None, None
best_test_y, best_test_prob = None, None
best_epoch = -1

for epoch in range(1, HAWKES_EPOCHS + 1):
    t0 = time.time()
    loss = hawkes_run_epoch(train_mode=True)
    val_y, val_prob, test_y, test_prob = hawkes_run_epoch(train_mode=False)
    val_thresh_e, val_oracle_f1 = best_threshold_f1(val_y, val_prob)
    val_f1_05  = (f1_score(val_y,  (val_prob  >= 0.5).astype(np.int64), pos_label=1, zero_division=0)
                  if len(val_y)  else 0.0)
    test_f1_05 = (f1_score(test_y, (test_prob >= 0.5).astype(np.int64), pos_label=1, zero_division=0)
                  if len(test_y) else 0.0)
    is_best = val_oracle_f1 > best_val_oracle_f1
    if is_best:
        best_val_oracle_f1 = val_oracle_f1
        best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_val_y, best_val_prob = val_y, val_prob
        best_test_y, best_test_prob = test_y, test_prob
        best_epoch = epoch
    marker = "  *" if is_best else ""
    print(f"epoch {epoch:2d}  loss={loss:.4f}  val F1@0.5={val_f1_05:.4f}  val F1@best={val_oracle_f1:.4f}  "
          f"test F1@0.5={test_f1_05:.4f}{marker}  ({time.time()-t0:.1f}s)")

# === Restore best epoch and report on its val/test predictions ===
print(f"\nBest epoch: {best_epoch}  (val F1@best-threshold={best_val_oracle_f1:.4f})")
if best_state_dict is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state_dict.items()})
val_y, val_prob = best_val_y, best_val_prob
test_y, test_prob = best_test_y, best_test_prob

# === Threshold calibration on val (best-epoch predictions) ===
best_threshold, _ = best_threshold_f1(val_y, val_prob)
print(f"Best threshold from val: {best_threshold:.4f}")
test_pred_calib = (test_prob >= best_threshold).astype(np.int64)
report(test_y, test_pred_calib, test_prob, "Hawkes (best epoch + threshold-calibrated)")

Hawkes subset size (all labelled wallets): 265,354
Hawkes splits: train_fit=220,483  val_for_threshold=36,747  test=110,242
Train events: pos=9,731  neg=169,770  illicit_weight=17.45
epoch  1  loss=0.3214  val F1@0.5=0.2776  val F1@best=0.5047  test F1@0.5=0.2567  *  (5.1s)
epoch  2  loss=0.2312  val F1@0.5=0.3697  val F1@best=0.6432  test F1@0.5=0.3425  *  (5.1s)
epoch  3  loss=0.1544  val F1@0.5=0.4866  val F1@best=0.6607  test F1@0.5=0.4650  *  (5.1s)
epoch  4  loss=0.1329  val F1@0.5=0.5759  val F1@best=0.6738  test F1@0.5=0.5586  *  (5.1s)
epoch  5  loss=0.1323  val F1@0.5=0.4428  val F1@best=0.7321  test F1@0.5=0.4160  *  (5.1s)
epoch  6  loss=0.1275  val F1@0.5=0.5494  val F1@best=0.7604  test F1@0.5=0.5262  *  (5.1s)
epoch  7  loss=0.1033  val F1@0.5=0.5860  val F1@best=0.7673  test F1@0.5=0.5658  *  (5.1s)
epoch  8  loss=0.0996  val F1@0.5=0.5904  val F1@best=0.7827  test F1@0.5=0.5707  *  (5.1s)
epoch  9  loss=0.0908  val F1@0.5=0.5753  val F1@best=0.8019  test F1@0.5=0.5570 

(0.8796881091617934, 0.9925988769921021)

## 6. Summary

All numbers below are illicit-class metrics on the **same 30% test slice** of the per-row 70/30 stratified random split (paper protocol).

Print a sorted summary at the bottom of the run; the cell below collates `results` populated by each model's `report(...)` call.

In [35]:
print(f"{'Model':40s}  {'illicit F1':>10s}  {'AUC':>8s}")
print("-" * 64)
for name, r in sorted(results.items(), key=lambda kv: -kv[1]["f1"]):
    print(f"{name:40s}  {r['f1']:>10.4f}  {r['auc']:>8.4f}")

print("\nNotes:")
print("- Per-row 70/30 stratified random split applied identically to RF, LR, Transformer, and Hawkes.")
print(f"- Embedder config: conv={EMBEDDER_CONV_TYPE}, num_layers={EMBEDDER_NUM_LAYERS}, hidden={EMBEDDER_HIDDEN_DIM}.")
print(f"- Hawkes config: hidden={HAWKES_HIDDEN_DIM}, window={HAWKES_WINDOW}, fuse={HAWKES_FUSE_MODE}, focal_gamma={HAWKES_FOCAL_GAMMA}.")
print("- The Hawkes model's threshold is calibrated on a 10% slice carved from train; baselines and Transformer use threshold=0.5.")

Model                                     illicit F1       AUC
----------------------------------------------------------------
RF                                            0.9539    0.9978
Hawkes (threshold-calibrated)                 0.8714    0.9914
Transformer+SAGE                              0.8697    0.9975
Transformer+GCN                               0.8565    0.9978
LR                                            0.3916    0.8461

Notes:
- Per-row 70/30 stratified random split applied identically to RF, LR, Transformer, and Hawkes.
- Embedder config: conv=sage, num_layers=3, hidden=128.
- Hawkes config: hidden=256, window=8, fuse=gated, focal_gamma=2.0.
- The Hawkes model's threshold is calibrated on a 10% slice carved from train; baselines and Transformer use threshold=0.5.
